# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

# Personalized Technical Tutor

This tool takes a technical question and generates an easy to understand explanation using:

1. GPT-5-nano through OpenAI API
2. Open Source Model running locally through Ollama

Both models receive the same question and same tutor instructions.

In [1]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


## Constants

It Stores the model names and local ollama endpoint.

GPT-5-nano will be called through OpenAI.

The open source model will be called locally through Ollama.

In [2]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'


## Set Up the Enviornment

The OpenAI API Key is loaded from the .env file.

OpenAI client connects to GPT-5-nano.

Ollama connects to Open source model running locally.

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

API key looks good so far


In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}

Please Explain:
- what is set comprehension does
- what book.get("author") does
- why the if condition is used
- what yield from does
- How duplicate authors are removed
- Why the order of the results may not be predictable.
"""

In [5]:
# New Question

question = """
A binary classification model produced the following confusion matrix:

- True Positives = 40
- False Positives = 10
- True Negatives = 45
- False Negatives = 5

Please calculate and explain:

- Accuracy
- Precision
- Recall
- F1-score

Show each formula, substitute the values, and give the final answer rounded to 2 decimal places.
"""

## Messages

The same System Prompt and Technical question wil be sent to both models.

This makes it easier to compare the explanation.

In [6]:
# Mesaages
messages = [
    {"role":"system", "content":"system_prompt"},
    {"role":"user", "content":question}
]

In [7]:
# Get gpt-5-nano to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response),
    display_id = display_handle.display_id
    )

Here are the metrics calculated from TP=40, FP=10, TN=45, FN=5.

1) Accuracy
- Formula: Accuracy = (TP + TN) / (TP + FP + TN + FN)
- Substitution: (40 + 45) / (40 + 10 + 45 + 5) = 85 / 100 = 0.85
- Result (2 decimals): 0.85

2) Precision
- Formula: Precision = TP / (TP + FP)
- Substitution: 40 / (40 + 10) = 40 / 50 = 0.80
- Result (2 decimals): 0.80

3) Recall
- Formula: Recall = TP / (TP + FN)
- Substitution: 40 / (40 + 5) = 40 / 45 ≈ 0.8889
- Result (2 decimals): 0.89

4) F1-score
- Formula: F1 = 2 * (Precision * Recall) / (Precision + Recall)
  (or using exact fractions: F1 = 2PR / (P + R))
- Using P = 0.80 and R = 0.89:
  F1 ≈ 2 * (0.80 * 0.89) / (0.80 + 0.89) ≈ 1.424 / 1.69 ≈ 0.842
- Result (2 decimals): 0.84

Summary:
- Accuracy: 0.85
- Precision: 0.80
- Recall: 0.89
- F1-score: 0.84

In [8]:
# Warm up Llama 3.2 - It was taking a lot of time to run, just for warm up

warmup = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role":"user", "content":"Hi"}
    ]
)

print("Llama 3.2 is ready")

Llama 3.2 is ready


In [9]:
# Get Llama 3.2 to answer
response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages
)

display(Markdown(response.choices[0].message.content))

To calculate the accuracy, precision, recall, and F1-score, we'll use the following formulas:

**Accuracy**
Accuracy = (TP + TN) / (TP + TN + FP + FN)

Where:
- TP: True Positives (40)
- TN: True Negatives (45)
- FP: False Positives (10)
- FN: False Negatives (5)

Substituting the values:
Accuracy = (40 + 45) / (40 + 45 + 10 + 5)
Accuracy = 85 / 100
Accuracy = 0.85

**Precision**
Precision = TP / (TP + FP)

Where:
- TP: True Positives (40)
- FP: False Positives (10)

Substituting the values:
Precision = 40 / (40 + 10)
Precision = 40 / 50
Precision = 0.80

**Recall**
Recall = TP / (TP + FN)

Where:
- TP: True Positives (40)
- FN: False Negatives (5)

Substituting the values:
Recall = 40 / (40 + 5)
Recall = 40 / 45
Recall = 0.89

**F1-score**
F1-score = 2 × (Precision × Recall) / (Precision + Recall)

Substituting the values:
F1-score = 2 × (0.80 × 0.89) / (0.80 + 0.89)
F1-score = 2 × 0.712 / 1.69
F1-score = 1.424 / 1.69
F1-score = 0.84

So, the results are:

- Accuracy: 0.85
- Precision: 0.80
- Recall: 0.89
- F1-score: 0.84

# Week 2 Upgrade - Conversation Context

The Personalized Technical Tutor is now improved by giving it the previous conversations as background information.

This helps the tutor understand the context before answering the new question.

## Previous Conversations

The last two conversations are stored as background information.

The tutor will use them before answering the new technical question.

In [10]:
# Previous Conversations

previous_conversation_1 = """
User: What is supervised learning?
Tutor: Supervised learning is a type of machine learning where a model learns from labeled data.
"""

previous_conversation_2 = """
User: What is overfitting?
Tutor: Overfitting happens when a model learns the training data too closely and does not perform well on new data.
"""

In [11]:
# New Question: Now a new technical question is asked based on the previous conversations.
question = """
How can I prevent overfitting when training a supervised machine learning model?

Please Explain:

- Why overfitting happens
- How train and validation data helps
- How regularization helps
- How cross validation helps
- How early stopping helps
"""

## User Prompt

The previous conversations and the new technical question are combined into one User Prompt.

This gives the tutor the conversation context before answering.

In [12]:
# User Prompt

user_prompt = f"""
Previous Conversation 1:
{previous_conversation_1}

Previous Conversation 2:
{previous_conversation_2}

New Technical Question:
{question}
"""

## System Prompt

The System Prompt defines how the Technical Tutor should explain the questions.

The same instructions will be used for both models.

In [13]:
system_prompt = """
You are a technical tutor for Data Science and AI.

Explain technical concepts in simple and clear steps.

Assume familiarity with Python, statistics, data analysis and machine learning.

When helpful:
- Explain the main idea first
- Break the answer into simple steps
- Use practical examples
- Explain formulas and calculations clearly
"""

## Messages

The same System Prompt and User Prompt will be sent to both models.

The User Prompt now contains the previous conversations and the new technical question.

In [14]:
# Messages

messages = [
    {"role":"system", "content":system_prompt},
    {"role":"user", "content":user_prompt}
]

In [15]:
# Get gpt-5-nano to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response),
    display_id = display_handle.display_id
    )

 overview
Overfitting happens when a model learns the training data too well (including its noise) and fails to generalize to new data. You can reduce overfitting by using a good train/validation setup, adding regularization, validating your choices with cross-validation, and stopping training at the right time (early stopping).

1) Why overfitting happens
- Model complexity vs. data: A very flexible model can fit many patterns, including noise, if you don’t have enough data.
- Noise in the data: If the training set contains random fluctuations, the model may treat them as real signals.
- Insufficient training data: With few samples, the model sees limited variability and may not generalize.
- Data leakage or improper preprocessing: If information from the test set leaks into training (or future data scales are learned from all data), you get overly optimistic training metrics.
- Imbalanced data or poorly chosen features: Some features may capture idiosyncrasies that don’t generalize.

2) How train and validation data helps
- Purpose:
  - Training set: model learns parameters.
  - Validation set: provides an unbiased estimate of generalization to tune hyperparameters and compare models.
- Practical steps:
  - Split your data into train and validation (and often a separate test set).
  - Use the validation performance to decide model complexity (e.g., how many trees in a forest, or the degree of a polynomial, or a neural network size).
  - Keep the test set completely separate until final evaluation.

3) How regularization helps
What regularization does: it discourages the model from becoming too complex, typically by penalizing large or numerous parameter values.

Common regularization methods:
- L2 regularization (ridge): adds a penalty proportional to the sum of squares of the weights.
  - Loss: J = (training loss) + lambda * sum(w_i^2)
  - Effect: Shrinks weights toward zero, reducing variance but usually not forcing exact zeros.
- L1 regularization (lasso): adds a penalty proportional to the sum of absolute values of the weights.
  - Loss: J = (training loss) + lambda * sum(|w_i|)
  - Effect: Can drive some weights to exactly zero, effectively performing feature selection.
- Elastic net: a mix of L1 and L2.
- In neural networks: weight decay is L2 on weights; dropout randomly drops units during training to reduce co-adaptation.
- How to use: add the penalty with a regularization strength (lambda or alpha) and choose it via validation.
- Practical tip: start with a small lambda and increase it if you notice your validation performance worsens due to high variance.

4) How cross-validation helps
What cross-validation does: gives a more reliable estimate of generalization and helps select hyperparameters more robustly than a single train/validation split.

Common approaches:
- k-fold cross-validation:
  - Split data into k equal folds. For each fold:
    - Train on the other k-1 folds.
    - Validate on the held-out fold.
  - Average the validation metrics across folds.
- Stratified k-fold (for classification): ensures each fold has similar class proportions.
- When to use:
  - Small datasets where a single split wastes data.
  - When tuning hyperparameters or comparing several models.
- Practical tips:
  - Use cross-validation scores to pick hyperparameters (e.g., regularization strength, model complexity).
  - Be mindful of computational cost (CV can be expensive for big models).

5) How early stopping helps
What it does: stops training when the model no longer improves on a validation set, preventing it from fitting noise in the training data.

How to use:
- Monitor a validation metric (e.g., validation loss or accuracy) after each training epoch.
- Set a patience parameter: how many consecutive epochs with no improvement you tolerate before stopping.
- Optionally restore the weights from the epoch with the best validation performance.
- When to use:
  - Especially useful for neural networks and other iterative learners.
  - Also helpful when training time is a concern, since it prevents overtraining.

Practical workflow (summary)
- Step 1: Split data into train/validation (and test for final evaluation). Ensure no leakage (e.g., scaling parameters learned only on training data).
- Step 2: Start with a simple baseline model and a reasonable default regularization (e.g., L2 with a small lambda).
- Step 3: Evaluate on the validation set. If there’s a large gap between train and validation performance, try regularization and/or more data.
- Step 4: Use cross-validation to robustly compare models or to tune hyperparameters (including regularization strength).
- Step 5: If training a neural network, or a model where you train for many epochs, apply early stopping using a validation set.
- Step 6: After selecting a model/hyperparameters, evaluate the final performance on a separate test set to estimate real-world generalization.

Concrete quick-start tips (with Python-like ideas)
- Basic train/validation split:
  - from sklearn.model_selection import train_test_split
  - X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
- Regularized linear models:
  - Ridge (L2): from sklearn.linear_model import Ridge; model = Ridge(alpha=1.0)
  - Lasso (L1): from sklearn.linear_model import Lasso; model = Lasso(alpha=0.001)
  - ElasticNet: from sklearn.linear_model import ElasticNet; model = ElasticNet(alpha=0.001, l1_ratio=0.5)
- Cross-validation:
  - from sklearn.model_selection import cross_val_score
  - scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
  - For classification, use scoring like 'accuracy' or 'roc_auc'
- Early stopping (neural nets, e.g., Keras):
  - from keras.callbacks import EarlyStopping
  - early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
  - model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, callbacks=[early_stop])
- Data leakage caution:
  - Always compute scaling (and any feature engineering) on training data only, then apply to validation/test.
  - Use pipelines to automate this separation.

If you’d like, tell me your data type (tabular, images, text), the model you’re using (e.g., logistic regression, random forest, neural network), and the size of your dataset. I can suggest concrete settings (lambda values, number of folds, early stopping patience) and provide a small example pipeline tailored to your situation.

In [16]:
# Reusing Llama 3.2 to answer, with streaming
stream = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(
        Markdown(response),
        display_id=display_handle.display_id
    )

**Preventing Overfitting in Supervised Machine Learning Models**

Overfitting is a common issue in supervised machine learning where a model becomes too specialized in fitting the training data, leading to poor performance on new, unseen data. Let's break down the reasons why overfitting happens and the techniques to prevent it.

**Why Overfitting Happens**

Overfitting occurs when a model has too many parameters and is able to fit the noise in the training data, rather than the underlying patterns. This can be due to several reasons:

* **Small training dataset**: If the training dataset is too small, the model may not have enough data to learn from, leading to overfitting.
* **Complex model**: Models with many parameters, such as deep neural networks, can easily overfit the training data.
* **Lack of regularization**: Without regularization techniques, models can overfit the training data.

**How Train and Validation Data Helps**

To prevent overfitting, it's essential to use both training and validation data. Here's how:

1. **Split data into training and validation sets**: Divide your dataset into two parts: training (e.g., 80%) and validation (e.g., 20%). The training data will be used to train the model, while the validation data will be used to evaluate its performance.
2. **Train the model on the training data**: Train the model on the training data, using the desired parameters and hyperparameters.
3. **Evaluate the model on the validation data**: Once the model is trained, evaluate its performance on the validation data. This will give you an idea of how well the model generalizes to new data.

**How Regularization Helps**

Regularization techniques can help prevent overfitting by adding a penalty term to the loss function, discouraging the model from fitting the noise in the training data. Common regularization techniques include:

* **L1 regularization**: Adds a term to the loss function that is proportional to the absolute value of the model's parameters.
* **L2 regularization**: Adds a term to the loss function that is proportional to the squared value of the model's parameters.

Example: `L2 regularization` in Python using Scikit-learn:
```python
from sklearn.linear_model import Ridge
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

model = Ridge(alpha=0.1)
model.fit(X_train, y_train)
```
**How Cross-Validation Helps**

Cross-validation is a technique that splits the dataset into multiple folds and evaluates the model's performance on each fold. This helps to:

* **Get a more accurate estimate of the model's performance**: By evaluating the model on multiple folds, you can get a better sense of its performance on unseen data.
* **Reduce overfitting**: Cross-validation helps to ensure that the model is not overfitting by evaluating its performance on multiple datasets.

Example: K-fold cross-validation in Python using Scikit-learn:
```python
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

kf = KFold(n_splits=5, shuffle=True)
model = Ridge()
for train_index, val_index in kf.split(X_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]
    model.fit(X_train_fold, y_train_fold)
    model.predict(X_val_fold)
```
**How Early Stopping Helps**

Early stopping is a technique that stops the training process when the model's performance on the validation data starts to degrade. This helps to prevent overfitting by stopping the training process before the model becomes too specialized in fitting the training data.

Example: Early stopping in Python using Scikit-learn:
```python
from sklearn.linear_model import Ridge
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

model = Ridge()
best_val_loss = float('inf')
for epoch in range(100):
    model.fit(X_train, y_train)
    val_loss = model.score(X_val, y_val)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
    else:
        break
```
By using these techniques, you can prevent overfitting in your supervised machine learning models and ensure that they generalize well to new data.